In [3]:
import os
import base64
import ollama
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# ==========================================
# 📄 PHASE 1: VISION PROMPT (The Eyes)
# ==========================================
CLINICAL_EXTRACTION_PROMPT = """
Analyze this medical X-ray with extreme precision for a downstream Reasoning LLM. 
Extract and list the following attributes in a structured, technical format:
1. BONE(S) INVOLVED: Identify specific bones.
2. FRACTURE PRESENCE: [Yes/No/Inconclusive].
3. MORPHOLOGY: (e.g., Transverse, Oblique, Spiral, Comminuted, Greenstick).
4. LOCATION: Specific segment (e.g., Intra-articular, Mid-shaft, Proximal).
5. DISPLACEMENT: Mention percentage and direction.
6. ANGULATION: Degree and direction if visible.
7. SOFT TISSUE: Note any significant swelling or joint effusion.
8. CONFIDENCE SCORE: 0-100% based on image clarity.
Provide ONLY the technical extraction. Do not provide patient advice.
"""

# ==========================================
# 📄 PHASE 2: REASONING PROMPT (The Brain)
# ==========================================
REASONING_SYSTEM_PROMPT = """
ACT AS: A Consultant Orthopedic Surgeon.
CONTEXT: You are reviewing a technical extraction report provided by a Radiology AI. 
YOU DO NOT HAVE ACCESS TO THE ORIGINAL IMAGE. Your task is to interpret the text-based 
clinical features to finalize a management plan.

REQUIRED REPORT SECTIONS:
1. CLINICAL SYNTHESIS: Summarize the findings into a concise diagnosis.
2. TRIAGE CATEGORY: Classify as [EMERGENT], [URGENT], or [ROUTINE] based on the data.
3. PATHOPHYSIOLOGY: Explain the implications of the morphology (e.g., why a 'Spiral' fracture suggests rotation).
4. STABILITY ASSESSMENT: Determine if the features indicate a mechanically unstable fracture.
5. SURGICAL VS. NON-SURGICAL: Provide a recommendation based on orthopedic standards.
"""

# ==========================================
# 📸 PHASE 1 FUNCTIONS: FEATURE EXTRACTION
# ==========================================
def get_groq_vision(image_path):
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode('utf-8')
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": [
            {"type": "text", "text": CLINICAL_EXTRACTION_PROMPT},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
        ]}],
        temperature=0.0
    )
    return response.choices[0].message.content

def get_local_vision(image_path):
    # Optimized for GTX 1650: Clear VRAM immediately
    response = ollama.chat(
        model='qwen3-vl:2b',
        keep_alive=0, 
        messages=[{'role': 'user', 'content': CLINICAL_EXTRACTION_PROMPT, 'images': [image_path]}]
    )
    return response['message']['content']

# ==========================================
# 🧠 PHASE 2 FUNCTIONS: CLINICAL REASONING
# ==========================================
def get_groq_reasoning(features):
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    # Using Llama-3-70b for high-speed cloud reasoning
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": REASONING_SYSTEM_PROMPT},
            {"role": "user", "content": f"INPUT FEATURES:\n{features}"}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

def get_local_reasoning(features):
    # Using gemma4:e4b as your local reasoning model
    response = ollama.chat(
        model='gemma4:e4b',
        keep_alive=0,
        messages=[
            {"role": "system", "content": REASONING_SYSTEM_PROMPT},
            {"role": "user", "content": f"INPUT FEATURES:\n{features}"}
        ]
    )
    return response['message']['content']

# ==========================================
# 🚀 THE MASTER PIPELINE (The "Main Chain")
# ==========================================
def nexmed_pipeline(image_path, vision_choice="Groq", reasoning_choice="Groq"):
    # 1. Vision Extraction
    print(f"👀 Phase 1: Extracting features using {vision_choice}...")
    if vision_choice == "Groq":
        features = get_groq_vision(image_path)
    else:
        features = get_local_vision(image_path)
    
    # 2. Reasoning Report
    print(f"🧠 Phase 2: Generating report using {reasoning_choice}...")
    if reasoning_choice == "Groq":
        report = get_groq_reasoning(features)
    else:
        report = get_local_reasoning(features)
        
    return features, report

if __name__ == "__main__":
    img = r"C:\Users\ishaa\Downloads\images (1).jpg"
    features, report = nexmed_pipeline(img, vision_choice="Groq", reasoning_choice="Groq")
    
    print("\n" + "="*50)
    print("📋 PHASE 1 FEATURES:\n", features)
    print("\n" + "="*50)
    print("👨‍⚕️ PHASE 2 REPORT:\n", report)

👀 Phase 1: Extracting features using Groq...
🧠 Phase 2: Generating report using Groq...

📋 PHASE 1 FEATURES:
 1. **BONE(S) INVOLVED**: 
   - Radius
   - Ulna

2. **FRACTURE PRESENCE**: 
   - Yes

3. **MORPHOLOGY**: 
   - Radius: Oblique fracture
   - Ulna: Oblique fracture

4. **LOCATION**: 
   - Radius: Mid-shaft
   - Ulna: Mid-shaft

5. **DISPLACEMENT**: 
   - Radius: Approximately 10-15% displacement in a dorsal direction
   - Ulna: Approximately 5-10% displacement in a dorsal direction

6. **ANGULATION**: 
   - Visible angulation in both bones, approximately 5 degrees in a dorsal direction.

7. **SOFT TISSUE**: 
   - No significant swelling or joint effusion noted.

8. **CONFIDENCE SCORE**: 
   - 90%

👨‍⚕️ PHASE 2 REPORT:
 **1. CLINICAL SYNTHESIS**  
- **Diagnosis:** Closed, diaphyseal (mid‑shaft) oblique fractures of the radius and ulna with mild dorsal displacement (≈10‑15 % radius, 5‑10 % ulna) and ~5° dorsal angulation. No soft‑tissue swelling, joint effusion, or neurovascular 

In [1]:
"""
==========================================================
  NexMed AI - main_chain.py
  Pure ML logic. No web server.
==========================================================
"""

import os
import re
import base64

import ollama
from groq import Groq
from dotenv import load_dotenv

load_dotenv()


# ==========================================================
# PHASE 1 PROMPT - VISION (The Eyes)
# ==========================================================
CLINICAL_EXTRACTION_PROMPT = """
Analyze this medical X-ray with extreme precision for a downstream Reasoning LLM. 
Extract and list the following attributes in a structured, technical format:
1. BONE(S) INVOLVED: Identify specific bones (e.g., Distal Radius, Fifth Metatarsal).
2. FRACTURE PRESENCE: [Yes/No/Inconclusive].
3. MORPHOLOGY: (e.g., Transverse, Oblique, Spiral, Comminuted, Greenstick).
4. LOCATION: Specific segment (e.g., Intra-articular, Mid-shaft, Proximal).
5. DISPLACEMENT: Mention percentage and direction (e.g., 2mm dorsal displacement).
6. ANGULATION: Degree and direction if visible.
7. SOFT TISSUE: Note any significant swelling or joint effusion.
8. CONFIDENCE SCORE: 0-100% based on image clarity.

Provide ONLY the technical extraction. Do not provide patient advice.
"""


# ==========================================================
# PHASE 2 PROMPT - REASONING (The Brain)
# ==========================================================
REASONING_SYSTEM_PROMPT = """
ACT AS: A Consultant Orthopedic Surgeon.
CONTEXT: You are reviewing a technical extraction report provided by a Radiology AI. 
YOU DO NOT HAVE ACCESS TO THE ORIGINAL IMAGE. Your task is to interpret the text-based 
clinical features to finalize a management plan.

INPUT DATA SOURCE: Automated Vision Extraction Report.

REQUIRED REPORT SECTIONS:
1. CLINICAL SYNTHESIS: Summarize the findings into a concise diagnosis.
2. TRIAGE CATEGORY: Classify as [EMERGENT], [URGENT], or [ROUTINE] based on the data.
3. PATHOPHYSIOLOGY: Explain the implications of the specific morphology described 
   (e.g., why a 'Spiral' fracture at the 'Mid-shaft' suggests a rotational injury).
4. STABILITY ASSESSMENT: Determine if the features (displacement/angulation) 
   indicate a mechanically unstable fracture.
5. SURGICAL VS. NON-SURGICAL: Provide a recommendation based on orthopedic standards.

TONE: Decisive, professional, and strictly data-driven.
FORMATTING: Begin each section on its own line, prefixed with the section number 
and the exact section title in uppercase followed by a colon (e.g., '1. CLINICAL SYNTHESIS:'). 
Use '- ' at the start of each bullet, one bullet per line. Do not inline multiple sections together.
"""


# ==========================================================
# PHASE 1 - VISION FUNCTIONS
# ==========================================================
def get_groq_vision(image_path):
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": CLINICAL_EXTRACTION_PROMPT},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
            ],
        }],
        temperature=0.0,
    )
    return response.choices[0].message.content


def get_local_vision(image_path):
    response = ollama.chat(
        model="qwen3-vl:2b",
        keep_alive=0,
        messages=[{
            "role": "user",
            "content": CLINICAL_EXTRACTION_PROMPT,
            "images": [image_path],
        }],
    )
    return response["message"]["content"]


def vision_extractor_factory(image_path, model_choice="Groq"):
    if model_choice == "Groq":
        return get_groq_vision(image_path)
    return get_local_vision(image_path)


# ==========================================================
# PHASE 2 - REASONING FUNCTIONS
# ==========================================================
def get_groq_reasoning(features_text):
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": REASONING_SYSTEM_PROMPT},
            {"role": "user",   "content": f"INPUT FEATURES:\n{features_text}"},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


def get_local_reasoning(features_text):
    response = ollama.chat(
        model="gemma4:e4b",
        keep_alive=0,
        messages=[
            {"role": "system", "content": REASONING_SYSTEM_PROMPT},
            {"role": "user",   "content": f"INPUT FEATURES:\n{features_text}"},
        ],
    )
    return response["message"]["content"]


def reasoning_factory(features_text, model_choice="Groq"):
    if model_choice == "Groq":
        return get_groq_reasoning(features_text)
    return get_local_reasoning(features_text)


# ==========================================================
# FULL PIPELINE (convenience chain for notebooks / CLI)
# ==========================================================
def nexmed_pipeline(image_path, vision_choice="Groq", reasoning_choice="Groq"):
    print(f"Phase 1: Extracting features using {vision_choice}...")
    features = vision_extractor_factory(image_path, vision_choice)

    print(f"Phase 2: Generating report using {reasoning_choice}...")
    report = reasoning_factory(features, reasoning_choice)

    return features, report


# ==========================================================
# MARKDOWN -> DICT PARSERS
# (robust to inline headers, bold, numbered, plain)
# ==========================================================
FEATURE_KEYS = [
    ("BONE",         r"BONE\(?S?\)?\s*INVOLVED"),
    ("FRACTURE",     r"FRACTURE\s*PRESENCE"),
    ("MORPHOLOGY",   r"MORPHOLOGY"),
    ("LOCATION",     r"LOCATION"),
    ("DISPLACEMENT", r"DISPLACEMENT"),
    ("ANGULATION",   r"ANGULATION"),
    ("SOFT TISSUE",  r"SOFT\s*TISSUE"),
    ("CONFIDENCE",   r"CONFIDENCE\s*SCORE"),
]

REPORT_KEYS = [
    ("CLINICAL SYNTHESIS",        r"CLINICAL\s*SYNTHESIS"),
    ("TRIAGE CATEGORY",           r"TRIAGE\s*CATEGORY"),
    ("PATHOPHYSIOLOGY",           r"PATHOPHYSIOLOGY"),
    ("STABILITY ASSESSMENT",      r"STABILITY\s*ASSESSMENT"),
    ("SURGICAL VS. NON-SURGICAL", r"SURGICAL\s*VS\.?\s*NON[- ]?SURGICAL"),
]


def _clean_value(text):
    """Strip markdown junk: **bold**, leading dashes/colons/bullets, extra whitespace."""
    text = re.sub(r"\*+", "", text)
    text = re.sub(r"^[\s\-\:•]+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _prettify_body(text):
    """
    Turn run-on bullet prose into readable multi-line output.

    The LLM often emits:  "- Point one. - Point two. - Point three."
    We convert inline ' - ' separators into real line breaks so each bullet
    sits on its own line. Same for ' • ' and numbered sub-points.
    """
    # Remove markdown bold/italic
    text = re.sub(r"\*+", "", text)

    # Normalize whitespace first (collapse runs, keep real newlines intact)
    text = re.sub(r"[ \t]+", " ", text)

    # Split inline bullet patterns onto their own lines.
    # Match ' - ' or ' — ' or ' • ' when used as a bullet separator
    # (i.e., preceded by end-of-sentence punctuation or another bullet).
    text = re.sub(r"\s+[-–—•]\s+", "\n- ", text)

    # Split inline numbered sub-points: " 1. ", " 2) " etc.  -> newline + number.
    text = re.sub(r"(?<=[.\]\)])\s+(\d+[\.\)])\s+", r"\n\1 ", text)

    # Clean accidental empty lines
    text = re.sub(r"\n\s*\n+", "\n", text).strip()

    return text


def parse_sections(raw_text, schema, prettify=False):
    """
    Robust section splitter.

    Builds ONE regex that finds any of the schema headers anywhere in the text
    (not just line-start) — so it catches inline headers the LLM sometimes
    emits ("...urgency. 3. PATHOPHYSIOLOGY - Green-stick..."). Returns an
    ordered dict {label: body}, with every schema key present.
    """
    combined = "|".join(f"(?P<k{i}>{pat})" for i, (_, pat) in enumerate(schema))
    # Header pattern accepts:
    #   - optional leading section number "1." / "1)"
    #   - optional ** bold markers **
    #   - the header text itself (from schema)
    #   - trailing colon / dash / en-dash / em-dash separator
    header_pattern = re.compile(
        rf"(?i)(?:(?<=^)|(?<=[\s.\]\)\-—–]))"   # start-of-string or sensible boundary
        rf"(?:\d+\s*[\.\)]\s*)?"                  # optional "1." / "1)"
        rf"\*{{0,2}}\s*"                          # optional **
        rf"(?:{combined})"                        # one of our headers
        rf"\s*\*{{0,2}}"                          # optional closing **
        rf"\s*[:\-–—]?\s*"                        # optional separator
    )

    matches = list(header_pattern.finditer(raw_text))
    result = {}

    if not matches:
        # No headers found -> dump everything under the first schema key
        if schema:
            body = _clean_value(raw_text) or "No data."
            result[schema[0][0]] = _prettify_body(body) if prettify else body
        for label, _ in schema:
            result.setdefault(label, "—")
        return {label: result[label] for label, _ in schema}

    for idx, match in enumerate(matches):
        matched_label = None
        for i, (label, _pat) in enumerate(schema):
            if match.group(f"k{i}"):
                matched_label = label
                break
        if matched_label is None:
            continue

        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(raw_text)
        body = raw_text[start:end].strip()

        # Strip wrapping bold / bullets but KEEP real newlines for bullet lists
        body = re.sub(r"\*+", "", body)
        body = re.sub(r"^[\s\-\:•]+", "", body)
        body = body.strip()

        if prettify:
            body = _prettify_body(body)
        else:
            # For short feature values we want a single tidy line
            body = re.sub(r"\s+", " ", body).strip()

        if not body:
            body = "—"

        # Keep the longer body if duplicates somehow slip through
        if matched_label in result and len(result[matched_label]) >= len(body):
            continue
        result[matched_label] = body

    for label, _ in schema:
        result.setdefault(label, "—")

    return {label: result[label] for label, _ in schema}


def parse_features(raw_text):
    # Features are short one-liners — no prettify
    return parse_sections(raw_text, FEATURE_KEYS, prettify=False)


def parse_report(raw_text):
    # Reports are long prose with bullets — prettify into multiline
    return parse_sections(raw_text, REPORT_KEYS, prettify=True)


# ==========================================================
# STANDALONE TEST
# ==========================================================
if __name__ == "__main__":
    test_path = r"C:\Users\ishaa\Downloads\images (1).jpg"

    features, report = nexmed_pipeline(
        test_path,
        vision_choice="Groq",
        reasoning_choice="Groq",
    )

    print("\n" + "=" * 50)
    print("PHASE 1 - FEATURES (raw):\n", features)
    print("\n" + "=" * 50)
    print("PHASE 2 - REPORT (raw):\n", report)
    print("\n" + "=" * 50)
    print("PARSED FEATURES (for UI):\n", parse_features(features))
    print("\n" + "=" * 50)
    print("PARSED REPORT (for UI):\n", parse_report(report))

Phase 1: Extracting features using Groq...
Phase 2: Generating report using Groq...

PHASE 1 - FEATURES (raw):
 **BONE(S) INVOLVED:** 
- Radius
- Ulna

**FRACTURE PRESENCE:** 
- Yes

**MORPHOLOGY:** 
- Both bones exhibit a similar fracture pattern.

**LOCATION:** 
- Radius: Mid-shaft
- Ulna: Mid-shaft

**DISPLACEMENT:** 
- Radius: Approximately 50% dorsal displacement
- Ulna: Approximately 40% dorsal displacement

**ANGULATION:** 
- Both fractures show an apex volar angulation, approximately 15 degrees.

**SOFT TISSUE:** 
- No significant swelling or joint effusion noted.

**CONFIDENCE SCORE:** 
- 90%

PHASE 2 - REPORT (raw):
 1. CLINICAL SYNTHESIS:  
- Displaced mid‑shaft fractures of the radius and ulna with ~50% dorsal displacement of the radius, ~40% dorsal displacement of the ulna, and ~15° volar apex angulation of both bones.  

2. TRIAGE CATEGORY:  
- [URGENT] – Requires operative management within 24 hours to restore forearm alignment and prevent loss of pronation/supination.  